In [ ]:
!pip install optuna shap xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 6.9 MB/s eta 0:00:00


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os
os.makedirs('/content', exist_ok=True)   # ensure output folder exists

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
import seaborn as sns
from scipy import stats

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score, GridSearchCV
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                              r2_score, mean_absolute_percentage_error)
from sklearn.inspection import permutation_importance

#Style
plt.rcParams.update({
    'font.family':'DejaVu Sans','font.size':11,'axes.titlesize':13,
    'axes.labelsize':12,'xtick.labelsize':10,'ytick.labelsize':10,
    'figure.dpi':150,'savefig.dpi':300,'savefig.bbox':'tight',
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.grid':True,'grid.alpha':0.3,'grid.linestyle':'--',
})
C = ['#2C7BB6','#D7191C','#1A9641','#FDAE61','#756BB1','#636363','#E6AB02']
MN = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
      7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
fmt = FuncFormatter(lambda x,_: f'{int(x):,}')
OUT = '/content'

print("="*70)
print("  DENGUE CLIMATE ANALYSIS — BANGLADESH")
print("  Random Forest Regression (Log-Transformed) | Train:2008–2024 | Test:2025")
print("="*70)

# 1. LOAD
df = pd.read_csv(f'{OUT}/Dengue_Climate_Bangladesh.csv')
df = df.sort_values(['YEAR','MONTH']).reset_index(drop=True)
print(f"\n[DATA] {df.shape} | Years: {df.YEAR.min()}–{df.YEAR.max()} | Missing: {df.isnull().sum().sum()}")

#2. FEATURE ENGINEERING
df['MONTH_SIN']  = np.sin(2*np.pi*df['MONTH']/12)
df['MONTH_COS']  = np.cos(2*np.pi*df['MONTH']/12)
df['TEMP_RANGE'] = df['MAX'] - df['MIN']
df['HEAT_INDEX'] = df['MAX'] + 0.33*df['HUMIDITY'] - 4.0

# Lag features (1–3 months)
for lag in [1,2,3]:
    for col in ['DENGUE','RAINFALL','HUMIDITY','MAX','MIN','TEMP_RANGE']:
        df[f'{col}_LAG{lag}'] = df[col].shift(lag)

# Rolling means (3-month window, shifted to avoid leakage)
for col in ['DENGUE','RAINFALL','HUMIDITY','MAX']:
    df[f'{col}_ROLL3'] = df[col].shift(1).rolling(3).mean()

# Accumulated rainfall
df['RAIN_CUM3'] = (df['RAINFALL_LAG1'].fillna(0) +
                   df['RAINFALL_LAG2'].fillna(0) +
                   df['RAINFALL_LAG3'].fillna(0))

df.dropna(inplace=True); df.reset_index(drop=True, inplace=True)

FEATURES = [
    'MIN','MAX','HUMIDITY','RAINFALL','TEMP_RANGE','HEAT_INDEX',
    'MONTH_SIN','MONTH_COS',
    'DENGUE_LAG1','DENGUE_LAG2','DENGUE_LAG3',
    'RAINFALL_LAG1','RAINFALL_LAG2','RAINFALL_LAG3',
    'HUMIDITY_LAG1','HUMIDITY_LAG2','HUMIDITY_LAG3',
    'MAX_LAG1','MAX_LAG2','MAX_LAG3',
    'MIN_LAG1','TEMP_RANGE_LAG1',
    'DENGUE_ROLL3','RAINFALL_ROLL3','HUMIDITY_ROLL3','MAX_ROLL3',
    'RAIN_CUM3',
]
TARGET = 'DENGUE'
print(f"[FEATURES] {len(FEATURES)} features | Final rows: {len(df)}")

#3. SPLIT
train = df[df['YEAR']<=2024].copy()
test  = df[df['YEAR']==2025].copy()

X_tr, y_tr = train[FEATURES], train[TARGET]
X_te, y_te = test[FEATURES],  test[TARGET]

# LOG-TRANSFORM target (log1p handles zeros)
y_tr_log = np.log1p(y_tr)
print(f"\n[SPLIT] Train: {len(X_tr)} | Test: {len(X_te)}")
print(f"[TRANSFORM] y_train range: {y_tr.min()}–{y_tr.max()} → log: {y_tr_log.min():.2f}–{y_tr_log.max():.2f}")

#4. HYPERPARAMETER TUNING
print("\n[TUNING] Grid Search with TimeSeriesSplit(5) on log-scale target...")
tscv = TimeSeriesSplit(n_splits=5)
param_grid = {
    'n_estimators':      [300, 500, 700],
    'max_depth':         [None, 15, 25],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', 0.7],
}
base_rf = RandomForestRegressor(random_state=42, n_jobs=-1, oob_score=True)
gs = GridSearchCV(base_rf, param_grid, cv=tscv,
                  scoring='neg_mean_absolute_error', n_jobs=-1, verbose=0)
gs.fit(X_tr, y_tr_log)
best = gs.best_params_
print(f"[TUNING] Best params: {best}")

#5. FINAL MODEL
rf = RandomForestRegressor(**best, random_state=42, n_jobs=-1, oob_score=True)
rf.fit(X_tr, y_tr_log)
print(f"[MODEL] OOB R² (log-scale): {rf.oob_score_:.4f}")

#6. CROSS-VALIDATION
cv_r2_log = cross_val_score(rf, X_tr, y_tr_log, cv=tscv, scoring='r2')
cv_mae_orig = []
for tr_idx, val_idx in tscv.split(X_tr):
    m = RandomForestRegressor(**best, random_state=42, n_jobs=-1)
    m.fit(X_tr.iloc[tr_idx], y_tr_log.iloc[tr_idx])
    cv_mae_orig.append(mean_absolute_error(y_tr.iloc[val_idx],
                                            np.expm1(m.predict(X_tr.iloc[val_idx]))))
cv_mae_orig = np.array(cv_mae_orig)
print(f"\n[CV — log R²]  : {cv_r2_log.mean():.4f} ± {cv_r2_log.std():.4f}")
print(f"[CV — orig MAE]: {cv_mae_orig.mean():.1f} ± {cv_mae_orig.std():.1f}")

#7. PREDICTIONS & METRICS
y_tr_pred_log  = rf.predict(X_tr)
y_tr_pred_orig = np.expm1(y_tr_pred_log)
y_te_pred_log  = rf.predict(X_te)
y_te_pred_orig = np.expm1(y_te_pred_log)

def metrics(y_true, y_pred, label):
    mae    = mean_absolute_error(y_true, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_true, y_pred))
    r2     = r2_score(y_true, y_pred)
    mape   = mean_absolute_percentage_error(y_true+1, y_pred+1)*100
    r      = np.corrcoef(y_true, y_pred)[0,1]
    nse    = 1 - np.sum((y_true-y_pred)**2)/np.sum((y_true-np.mean(y_true))**2)
    r2_log = r2_score(np.log1p(y_true), np.log1p(np.maximum(y_pred,0)))
    r_log  = np.corrcoef(np.log1p(y_true), np.log1p(np.maximum(y_pred,0)))[0,1]
    print(f"\n  [{label}]")
    print(f"    MAE:       {mae:.2f}  |  RMSE:  {rmse:.2f}")
    print(f"    R² (orig): {r2:.4f}  |  R² (log): {r2_log:.4f}")
    print(f"    Pearson r (orig): {r:.4f}  |  r (log): {r_log:.4f}")
    print(f"    MAPE: {mape:.2f}%  |  NSE: {nse:.4f}")
    return dict(MAE=mae, RMSE=rmse, R2=r2, R2_log=r2_log,
                r=r, r_log=r_log, MAPE=mape, NSE=nse)

print("\n"+"="*55)
print("  MODEL PERFORMANCE METRICS")
print("="*55)
m_tr = metrics(y_tr, y_tr_pred_orig, "TRAIN 2008–2024")
m_te = metrics(y_te, y_te_pred_orig, "TEST  2025 (Jan–Oct)")

#8. PREDICTION TABLE
pt = test[['YEAR','MONTH']].copy()
pt['Month']         = pt['MONTH'].map(MN)
pt['Actual']        = y_te.values
pt['Predicted']     = y_te_pred_orig.round(0).astype(int)
pt['Abs_Error']     = (pt['Actual'] - pt['Predicted']).abs()
pt['Rel_Error_%']   = (pt['Abs_Error']/(pt['Actual']+1)*100).round(2)
pt['Log_Actual']    = np.log1p(pt['Actual']).round(3)
pt['Log_Predicted'] = np.log1p(pt['Predicted']).round(3)
print("\n"+"="*70)
print("  2025 PREDICTION TABLE (Jan–Oct)")
print("="*70)
print(pt[['Month','Actual','Predicted','Abs_Error','Rel_Error_%',
          'Log_Actual','Log_Predicted']].to_string(index=False))

#Feature importance
perm = permutation_importance(rf, X_te, np.log1p(y_te),
                               n_repeats=30, random_state=42, n_jobs=-1)
fi = pd.DataFrame({'Feature': FEATURES,
                   'MDI':       rf.feature_importances_,
                   'Perm_mean': perm.importances_mean,
                   'Perm_std':  perm.importances_std,
                  }).sort_values('MDI', ascending=False).reset_index(drop=True)
print("\n[TOP-10 FEATURES (MDI)]")
print(fi.head(10)[['Feature','MDI','Perm_mean']].to_string(index=False))
mnlbs = pt['Month'].tolist()

# FIGURE 1 — MAIN RESULTS DASHBOARD
fig = plt.figure(figsize=(18, 15))
gs0 = gridspec.GridSpec(3, 3, figure=fig, hspace=0.48, wspace=0.38)

ax1 = fig.add_subplot(gs0[0,:])
n_tr = len(y_tr)
xtr  = range(n_tr)
xte  = range(n_tr, n_tr+len(y_te))
ax1.plot(xtr, y_tr.values,     color=C[5], lw=1.0, label='Actual (Train 2008–2024)', alpha=0.75)
ax1.plot(xtr, y_tr_pred_orig,  color=C[0], lw=1.0, ls='--', label='Predicted (Train)', alpha=0.75)
ax1.plot(xte, y_te.values,     color=C[1], lw=2.5, marker='o', ms=7, label='Actual (2025 Jan–Oct)', zorder=5)
ax1.plot(xte, y_te_pred_orig,  color=C[2], lw=2.5, marker='s', ms=7, ls='--', label='Predicted (2025 Jan–Oct)', zorder=5)
ax1.fill_between(xte, y_te.values, y_te_pred_orig, alpha=0.15, color=C[3], label='Error band')
ax1.axvline(n_tr-0.5, color='black', ls=':', lw=1.5)
ax1.text(n_tr-0.5, ax1.get_ylim()[1]*0.85, '  Train / Test', fontsize=9, color='black')
xtk = list(range(0, n_tr, 12)) + list(xte)
xlb = [f"{int(train.iloc[i]['YEAR'])}-{MN[int(train.iloc[i]['MONTH'])]}"
       if i < n_tr
       else f"{int(test.iloc[i-n_tr]['YEAR'])}-{MN[int(test.iloc[i-n_tr]['MONTH'])]}"
       for i in xtk]
ax1.set_xticks(xtk); ax1.set_xticklabels(xlb, rotation=45, ha='right', fontsize=8)
ax1.set_ylabel('Dengue Cases'); ax1.yaxis.set_major_formatter(fmt)
ax1.set_title('A.  Actual vs. Predicted Dengue Cases — Full Timeline (2008–2025)\n'
              '(Log-transformed training; back-transformed for display)', fontweight='bold')
ax1.legend(loc='upper left', fontsize=9, framealpha=0.7)

ax2 = fig.add_subplot(gs0[1,:2])
x = np.arange(len(mnlbs)); w = 0.38
b1 = ax2.bar(x-w/2, y_te.values,    w, color=C[1], label='Actual 2025',    edgecolor='k', lw=0.6, alpha=0.9)
b2 = ax2.bar(x+w/2, y_te_pred_orig, w, color=C[2], label='Predicted 2025', edgecolor='k', lw=0.6, alpha=0.9)
for bar in b1:
    ax2.text(bar.get_x()+bar.get_width()/2., bar.get_height(),
             f'{int(bar.get_height()):,}', ha='center', va='bottom', fontsize=7, rotation=45)
for bar in b2:
    ax2.text(bar.get_x()+bar.get_width()/2., bar.get_height(),
             f'{int(bar.get_height()):,}', ha='center', va='bottom',
             fontsize=7, rotation=45, color=C[2])
ax2.set_xticks(x); ax2.set_xticklabels(mnlbs)
ax2.set_ylabel('Dengue Cases'); ax2.yaxis.set_major_formatter(fmt)
ax2.set_title('B.  2025 Monthly Actual vs. Predicted Dengue Cases', fontweight='bold')
ax2.legend()

ax3 = fig.add_subplot(gs0[1,2])
log_act  = np.log1p(y_te.values)
log_pred = y_te_pred_log
sc = ax3.scatter(log_act, log_pred, c=range(len(y_te)),
                  cmap='plasma', s=120, zorder=5, edgecolors='k', lw=0.5)
lml = max(log_act.max(), log_pred.max())*1.1
ax3.plot([0,lml],[0,lml],'r--',lw=1.5,label='1:1 line')
ax3.set_xlim(0,lml); ax3.set_ylim(0,lml)
ax3.set_xlabel('log(Actual + 1)'); ax3.set_ylabel('log(Predicted + 1)')
ax3.set_title('C.  Log-Scale Scatter (2025)', fontweight='bold')
plt.colorbar(sc, ax=ax3, label='Month (Jan=0)')
for i,(la,lp,ml) in enumerate(zip(log_act, log_pred, mnlbs)):
    ax3.annotate(ml, (la,lp), textcoords='offset points', xytext=(5,3), fontsize=8)
ax3.text(0.05, 0.88, f'R²(log)={m_te["R2_log"]:.3f}\nr(log)={m_te["r_log"]:.3f}',
          transform=ax3.transAxes, fontsize=10, bbox=dict(facecolor='wheat',alpha=0.7))
ax3.legend()

ax4 = fig.add_subplot(gs0[2,:2])
res = y_te.values - y_te_pred_orig
ax4.bar(x, res, color=[C[2] if v>=0 else C[1] for v in res], edgecolor='k', lw=0.5, alpha=0.85)
ax4.axhline(0, color='k', lw=1.2)
ax4.set_xticks(x); ax4.set_xticklabels(mnlbs)
ax4.set_ylabel('Residual (Actual − Predicted)'); ax4.yaxis.set_major_formatter(fmt)
ax4.set_title('D.  Residuals by Month (2025 Test Set)', fontweight='bold')

ax5 = fig.add_subplot(gs0[2,2]); ax5.axis('off')
td = [['Metric','Train\n(2008–24)','Test\n(2025)'],
      ['MAE',           f"{m_tr['MAE']:.0f}",        f"{m_te['MAE']:.0f}"],
      ['RMSE',          f"{m_tr['RMSE']:.0f}",       f"{m_te['RMSE']:.0f}"],
      ['R² (orig)',     f"{m_tr['R2']:.4f}",         f"{m_te['R2']:.4f}"],
      ['R² (log)',      f"{m_tr['R2_log']:.4f}",     f"{m_te['R2_log']:.4f}"],
      ['Pearson r(log)',f"{m_tr['r_log']:.4f}",      f"{m_te['r_log']:.4f}"],
      ['MAPE',          f"{m_tr['MAPE']:.1f}%",      f"{m_te['MAPE']:.1f}%"],
      ['NSE',           f"{m_tr['NSE']:.4f}",        f"{m_te['NSE']:.4f}"],
      ['OOB R²(log)',   f"{rf.oob_score_:.4f}",      '–'],
      ['CV R²(log)',    f"{cv_r2_log.mean():.4f}±{cv_r2_log.std():.4f}", '–'],
      ['CV MAE',        f"{cv_mae_orig.mean():.0f}±{cv_mae_orig.std():.0f}", '–'],
     ]
tbl = ax5.table(cellText=td[1:], colLabels=td[0],
                cellLoc='center', loc='center', bbox=[0,0.02,1,0.98])
tbl.auto_set_font_size(False); tbl.set_fontsize(8.5)
for (r,c),cell in tbl.get_celld().items():
    if r==0: cell.set_facecolor('#2C7BB6'); cell.set_text_props(color='white',fontweight='bold')
    elif r%2==0: cell.set_facecolor('#EFF3FF')
    cell.set_edgecolor('grey')
ax5.set_title('E.  Model Performance Metrics', fontweight='bold', pad=12)

fig.suptitle('Random Forest Regression: Dengue Case Forecasting in Bangladesh (2008–2025)\n'
             f'Features: {len(FEATURES)} | Log-transformed target | Best params: {best}',
             fontsize=12, fontweight='bold', y=1.01)
plt.savefig(f'{OUT}/Fig1_Main_Results.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n[SAVED] Fig1_Main_Results.png")
plt.close()

# FIGURE 2 — LOG-SCALE EVALUATION
fig2, axes2 = plt.subplots(1, 2, figsize=(16, 6))
fig2.suptitle('Log-Scale Model Evaluation (2025 Test Set)', fontsize=13, fontweight='bold')

ax2a = axes2[0]
sc2 = ax2a.scatter(log_act, log_pred, c=range(len(y_te)),
                    cmap='plasma', s=120, zorder=5, edgecolors='k', lw=0.5)
lml2 = max(log_act.max(), log_pred.max())*1.05
ax2a.plot([0,lml2],[0,lml2],'r--',lw=1.5,label='1:1 line')
ax2a.set_xlim(0,lml2); ax2a.set_ylim(0,lml2)
ax2a.set_xlabel('log(Actual Cases + 1)'); ax2a.set_ylabel('log(Predicted Cases + 1)')
ax2a.set_title('A.  Log-Scale Scatter (2025)', fontweight='bold')
plt.colorbar(sc2, ax=ax2a, label='Month (Jan=0)')
ax2a.legend()
ax2a.text(0.05, 0.88, f'R²={m_te["R2_log"]:.3f}\nr={m_te["r_log"]:.3f}',
           transform=ax2a.transAxes, fontsize=10, bbox=dict(facecolor='wheat',alpha=0.7))
for i,(la,lp,ml) in enumerate(zip(log_act, log_pred, mnlbs)):
    ax2a.annotate(ml, (la,lp), textcoords='offset points', xytext=(5,3), fontsize=8)

ax2b = axes2[1]
ax2b.plot(mnlbs, log_act,  'o-', color=C[1], lw=2.5, ms=8, label='log(Actual+1)')
ax2b.plot(mnlbs, log_pred, 's--', color=C[2], lw=2.5, ms=8, label='log(Predicted+1)')
ax2b.fill_between(range(len(mnlbs)), log_act, log_pred, alpha=0.2, color=C[3])
ax2b.set_xlabel('Month (2025)'); ax2b.set_ylabel('log(Cases + 1)')
ax2b.set_title('B.  Log-Scale Line Comparison (2025)', fontweight='bold')
ax2b.legend()

plt.tight_layout()
plt.savefig(f'{OUT}/Fig2_Log_Scale_Evaluation.png', dpi=300, bbox_inches='tight')
plt.show(); print("[SAVED] Fig2_Log_Scale_Evaluation.png"); plt.close()

# FIGURE 3 — FEATURE IMPORTANCE (MDI + Permutation)
fig3, axes3 = plt.subplots(1, 2, figsize=(16, 8))
fig3.suptitle('Feature Importance Analysis — Random Forest Regression', fontsize=13, fontweight='bold')

tn = 15
cmap_bar = plt.cm.RdYlBu_r(np.linspace(0.1, 0.9, tn))
fi_top_mdi = fi.head(tn).sort_values('MDI')
axes3[0].barh(fi_top_mdi['Feature'], fi_top_mdi['MDI'],
               color=cmap_bar, edgecolor='k', lw=0.5)
axes3[0].set_xlabel('Mean Decrease in Impurity (MDI)')
axes3[0].set_title(f'A.  MDI Importance (Top {tn})', fontweight='bold')
for i,(val,_) in enumerate(zip(fi_top_mdi['MDI'], fi_top_mdi['Feature'])):
    axes3[0].text(val+0.001, i, f'{val:.4f}', va='center', fontsize=8)

fi_top_perm = fi.sort_values('Perm_mean', ascending=False).head(tn).sort_values('Perm_mean')
axes3[1].barh(fi_top_perm['Feature'], fi_top_perm['Perm_mean'],
               xerr=fi_top_perm['Perm_std'], color=cmap_bar, edgecolor='k', lw=0.5,
               error_kw=dict(ecolor='gray', capsize=3, lw=1.2))
axes3[1].set_xlabel('Mean Decrease in R² (Permutation, 30 repeats)')
axes3[1].set_title(f'B.  Permutation Importance (Top {tn})', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{OUT}/Fig3_Feature_Importance.png', dpi=300, bbox_inches='tight')
plt.show(); print("[SAVED] Fig3_Feature_Importance.png"); plt.close()

# FIGURE 4 — CLIMATE–DENGUE EDA
fig4, axes4 = plt.subplots(2, 3, figsize=(18, 10))
fig4.suptitle('Exploratory Analysis: Climate Predictors vs. Dengue Incidence\n'
              'Training Set 2008–2024  |  log(Dengue+1) scale', fontsize=13, fontweight='bold')

cvars = [('RAINFALL','Rainfall (mm)',C[0]),('HUMIDITY','Humidity (%)',C[1]),
         ('MAX','Max Temp (°C)',C[2]),('MIN','Min Temp (°C)',C[3]),('TEMP_RANGE','Temp Range (°C)',C[4])]
for ax, (var, lbl, col) in zip(axes4.flatten(), cvars):
    xv = train[var]; yv = np.log1p(train[TARGET])
    ax.scatter(xv, yv, alpha=0.45, color=col, edgecolors='none', s=30)
    z = np.polyfit(xv, yv, 1); p = np.poly1d(z)
    xl = np.linspace(xv.min(), xv.max(), 100)
    ax.plot(xl, p(xl), 'k--', lw=1.5)
    r_sp, _ = stats.spearmanr(xv, train[TARGET])
    r_pe    = np.corrcoef(xv, yv)[0,1]
    ax.set_xlabel(lbl); ax.set_ylabel('log(Dengue Cases + 1)')
    ax.set_title(f'{var}  |  Pearson r={r_pe:.3f}  |  Spearman ρ={r_sp:.3f}', fontweight='bold')

ax_m = axes4.flatten()[5]
ma = train.groupby('MONTH')[TARGET].mean()
ax_m.bar(ma.index, ma.values,
          color=plt.cm.Spectral(np.linspace(0,1,12)), edgecolor='k', lw=0.5)
ax_m.set_xticks(range(1,13)); ax_m.set_xticklabels([MN[m] for m in range(1,13)], rotation=45)
ax_m.set_ylabel('Mean Dengue Cases'); ax_m.yaxis.set_major_formatter(fmt)
ax_m.set_title('Monthly Average Dengue Cases (2008–2024)', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{OUT}/Fig4_Climate_Dengue_EDA.png', dpi=300, bbox_inches='tight')
plt.show(); print("[SAVED] Fig4_Climate_Dengue_EDA.png"); plt.close()

# FIGURE 5 — CORRELATION HEATMAP
fig5, ax5h = plt.subplots(figsize=(11, 9))
core = ['DENGUE','RAINFALL','HUMIDITY','MAX','MIN','TEMP_RANGE',
        'DENGUE_LAG1','DENGUE_LAG2','DENGUE_LAG3',
        'RAINFALL_LAG1','RAINFALL_ROLL3','DENGUE_ROLL3','RAIN_CUM3','HEAT_INDEX']
cm = train[core].corr()
mask = np.triu(np.ones_like(cm, dtype=bool))
sns.heatmap(cm, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, ax=ax5h, linewidths=0.5, linecolor='white',
            annot_kws={'size':8})
ax5h.set_title('Correlation Matrix — Dengue & Climate Variables (Train Set 2008–2024)',
               fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(f'{OUT}/Fig5_Correlation_Heatmap.png', dpi=300, bbox_inches='tight')
plt.show(); print("[SAVED] Fig5_Correlation_Heatmap.png"); plt.close()

# FIGURE 6 — MODEL DIAGNOSTICS
fig6, axes6 = plt.subplots(2, 3, figsize=(18, 10))
fig6.suptitle('Model Diagnostics & Cross-Validation', fontsize=13, fontweight='bold')

fold_n = np.arange(1,6)
axes6[0,0].bar(fold_n, cv_r2_log, color=C[0], edgecolor='k', lw=0.5, alpha=0.85)
axes6[0,0].axhline(cv_r2_log.mean(), color=C[1], lw=2, ls='--',
                    label=f'Mean R²={cv_r2_log.mean():.3f}')
axes6[0,0].set_xlabel('CV Fold'); axes6[0,0].set_ylabel('R² (log-scale)')
axes6[0,0].set_title('A.  Cross-Val R² (TimeSeriesSplit, log-scale)', fontweight='bold')
axes6[0,0].legend(); axes6[0,0].set_xticks(fold_n)

axes6[0,1].bar(fold_n, cv_mae_orig, color=C[1], edgecolor='k', lw=0.5, alpha=0.85)
axes6[0,1].axhline(cv_mae_orig.mean(), color=C[0], lw=2, ls='--',
                    label=f'Mean MAE={cv_mae_orig.mean():.0f}')
axes6[0,1].set_xlabel('CV Fold'); axes6[0,1].set_ylabel('MAE (original scale)')
axes6[0,1].set_title('B.  Cross-Val MAE (original scale)', fontweight='bold')
axes6[0,1].legend(); axes6[0,1].set_xticks(fold_n); axes6[0,1].yaxis.set_major_formatter(fmt)

log_res_tr = y_tr_log.values - y_tr_pred_log
axes6[0,2].hist(log_res_tr, bins=25, color=C[0], edgecolor='k', lw=0.4, alpha=0.8)
axes6[0,2].axvline(0, color='red', lw=1.5, ls='--')
axes6[0,2].set_xlabel('Residual (log-scale)'); axes6[0,2].set_ylabel('Frequency')
axes6[0,2].set_title('C.  Residual Distribution — Train (log-scale)', fontweight='bold')
axes6[0,2].text(0.65, 0.9, f'μ={log_res_tr.mean():.3f}\nσ={log_res_tr.std():.3f}',
                 transform=axes6[0,2].transAxes, fontsize=9, bbox=dict(facecolor='wheat',alpha=0.7))

axes6[1,0].scatter(y_tr_pred_log, log_res_tr, alpha=0.35, color=C[0], s=20, edgecolors='none')
axes6[1,0].axhline(0, color='red', lw=1.5, ls='--')
axes6[1,0].set_xlabel('Fitted (log-scale)'); axes6[1,0].set_ylabel('Residuals (log-scale)')
axes6[1,0].set_title('D.  Residuals vs. Fitted (Train, log-scale)', fontweight='bold')

(osm, osr), (slope, intercept, rqq) = stats.probplot(log_res_tr, dist='norm', plot=None)
axes6[1,1].scatter(osm, osr, color=C[0], s=20, alpha=0.6, edgecolors='none')
axes6[1,1].plot(osm, slope*np.array(osm)+intercept, 'r--', lw=1.5)
axes6[1,1].set_xlabel('Theoretical Quantiles'); axes6[1,1].set_ylabel('Sample Quantiles')
axes6[1,1].set_title(f'E.  Q-Q Plot (Train Residuals)  |  r={rqq:.4f}', fontweight='bold')

yr_actual = train.groupby('YEAR')[TARGET].sum()
yr_pred_df = train.copy(); yr_pred_df['PRED'] = y_tr_pred_orig
yr_pred_agg = yr_pred_df.groupby('YEAR')['PRED'].sum()
axes6[1,2].plot(yr_actual.index, yr_actual.values, 'o-', color=C[1], lw=2, ms=6, label='Actual')
axes6[1,2].plot(yr_pred_agg.index, yr_pred_agg.values, 's--', color=C[2], lw=2, ms=6, label='Predicted')
axes6[1,2].set_xlabel('Year'); axes6[1,2].set_ylabel('Annual Dengue Cases')
axes6[1,2].set_title('F.  Annual Aggregated: Actual vs. Predicted (Train)', fontweight='bold')
axes6[1,2].legend(); axes6[1,2].yaxis.set_major_formatter(fmt)
plt.setp(axes6[1,2].xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
plt.savefig(f'{OUT}/Fig6_Diagnostics.png', dpi=300, bbox_inches='tight')
plt.show(); print("[SAVED] Fig6_Diagnostics.png"); plt.close()

# FIGURE 7 — YEARLY TREND & 2025 BURDEN
fig7, axes7 = plt.subplots(1, 2, figsize=(16, 6))
fig7.suptitle('Dengue Burden & 2025 Comparison', fontsize=13, fontweight='bold')

yr_tot = df.groupby('YEAR')[TARGET].sum().reset_index()
clrs   = [C[0] if y<2025 else C[1] for y in yr_tot['YEAR']]
axes7[0].bar(yr_tot['YEAR'], yr_tot[TARGET], color=clrs, edgecolor='k', lw=0.5, alpha=0.9)
axes7[0].set_xlabel('Year'); axes7[0].set_ylabel('Total Annual Dengue Cases')
axes7[0].set_title('A.  Annual Dengue Burden (2008–2025)', fontweight='bold')
axes7[0].yaxis.set_major_formatter(fmt)
axes7[0].set_xticks(yr_tot['YEAR']); plt.setp(axes7[0].xaxis.get_majorticklabels(), rotation=45)
from matplotlib.patches import Patch
axes7[0].legend(handles=[Patch(color=C[0],label='Train 2008–2024'),
                           Patch(color=C[1],label='Test 2025')], loc='upper left')

x2 = np.arange(len(mnlbs))
axes7[1].plot(mnlbs, y_te.values,    'o-',  color=C[1], lw=2.5, ms=8, label='Actual 2025')
axes7[1].plot(mnlbs, y_te_pred_orig, 's--', color=C[2], lw=2.5, ms=8, label='RF Predicted 2025')
axes7[1].fill_between(x2, y_te.values, y_te_pred_orig, alpha=0.2, color=C[3])
for i,(a,p) in enumerate(zip(y_te.values, y_te_pred_orig)):
    axes7[1].annotate(f'{int(p):,}', (i,p), textcoords='offset points',
                       xytext=(0,8), ha='center', fontsize=8, color=C[2])
axes7[1].set_xlabel('Month (2025)'); axes7[1].set_ylabel('Dengue Cases')
axes7[1].set_title('B.  2025 Monthly Actual vs. RF Predicted', fontweight='bold')
axes7[1].legend(); axes7[1].yaxis.set_major_formatter(fmt)

plt.tight_layout()
plt.savefig(f'{OUT}/Fig7_Annual_Trend.png', dpi=300, bbox_inches='tight')
plt.show(); print("[SAVED] Fig7_Annual_Trend.png"); plt.close()

#SAVE CSV TABLES
pt.to_csv(f'{OUT}/Table1_2025_Predictions.csv', index=False)
fi.to_csv(f'{OUT}/Table2_Feature_Importance.csv', index=False)

metrics_out = pd.DataFrame({
    'Metric':              ['MAE','RMSE','R² (orig)','R² (log)','Pearson r (log)',
                            'MAPE (%)','NSE','OOB R² (log)',
                            'CV R² (log) mean','CV R² (log) std'],
    'Train (2008–2024)':   [round(m_tr['MAE'],1), round(m_tr['RMSE'],1),
                            round(m_tr['R2'],4),  round(m_tr['R2_log'],4),
                            round(m_tr['r_log'],4),round(m_tr['MAPE'],2),
                            round(m_tr['NSE'],4), round(rf.oob_score_,4),
                            round(cv_r2_log.mean(),4), round(cv_r2_log.std(),4)],
    'Test 2025 (Jan–Oct)': [round(m_te['MAE'],1), round(m_te['RMSE'],1),
                            round(m_te['R2'],4),  round(m_te['R2_log'],4),
                            round(m_te['r_log'],4),round(m_te['MAPE'],2),
                            round(m_te['NSE'],4), '–', '–', '–'],
})
metrics_out.to_csv(f'{OUT}/Table3_Performance_Metrics.csv', index=False)

print("\n[SAVED] Table1_2025_Predictions.csv")
print("[SAVED] Table2_Feature_Importance.csv")
print("[SAVED] Table3_Performance_Metrics.csv")
print("\n"+"="*70)
print(f"  ALL OUTPUTS SAVED TO {OUT}/")
print("="*70)

  DENGUE CLIMATE ANALYSIS — BANGLADESH
  Random Forest Regression (Log-Transformed) | Train:2008–2024 | Test:2025

[DATA] (214, 7) | Years: 2008–2025 | Missing: 0
[FEATURES] 27 features | Final rows: 211

[SPLIT] Train: 201 | Test: 10
[TRANSFORM] y_train range: 0–79598 → log: 0.00–11.28

[TUNING] Grid Search with TimeSeriesSplit(5) on log-scale target...
